# SGLang 两处离群值：重复跑 3 轮，判定抖动还是真实

## 要交代的两个点

2026-09-05 那轮 SGLang（RadixAttention 开）出现两个说不清的数，当时如实记为
「单次测量、未复现、无解释，**不要拿它下结论**」。现在来把它定性。

| 位置 | 当时的数 | 为什么可疑 |
|---|---|---|
| 并发扫描 · 并发 16 | 吞吐 **496.0** tok/s，墙钟 16.51s，TPOT 30.64ms | 低于两侧（并发 8 = 1115.8，并发 32 = 902.4），墙钟异常长 |
| 稳态测试 · 一次性 3 条 | 吞吐 **22.1** tok/s，墙钟 6.33s | 邻居都是 0.4s 量级，但 **TPOT 正常（6.53ms）**——说明慢在解码之外 |

## 判据（跑之前写死，不许事后改）

每项重复 **3 轮**，看中位数与极差：

- **若并发 16 的吞吐三轮都显著低于并发 8 与 32** → 是**真实凹陷**，
  那它比一个漂亮的单调曲线更有价值，值得单独解释（调度、chunked prefill 边界等）。
- **若三轮里只有个别轮次塌、其余正常** → 是**抖动**（Colab 邻居干扰或一次性调度打嗝），
  当时那个数就是噪声，结论里应当剔除并说明。
- **稳态 3 条同理。** 它 TPOT 正常而墙钟长，若可复现，方向是「排队/调度」而非「解码慢」。

**两种结果都照实写。** 若是抖动，就等于承认当时那个数没有信息量。


## 0. 环境

In [ ]:
import subprocess
out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,compute_cap",
                      "--format=csv"], capture_output=True, text=True).stdout
print(out)
print("必须也是 Tesla T4，才与 2026-09-05 那轮可比。")


## 1. 装 SGLang —— 四个已知阻塞的修法

与 `cloud_sglang_v2.ipynb` 第 1 节完全相同（2026-09-05 实测得出）：
卸 `kernels`、卸 `torchaudio`、钉 `transformers==5.12.1`、不用 `sglang[all]`。


In [ ]:
import importlib.metadata as md_, subprocess, sys, os, shutil

CHECK = ["sglang", "aiohttp", "torchvision"]
TF_PIN = "transformers==5.12.1"

def ver(p):
    try:
        return md_.version(p)
    except Exception:
        return None

def sh(cmd):
    return subprocess.run(cmd, shell=isinstance(cmd, str), capture_output=True, text=True)

missing = [p for p in CHECK if ver(p) is None]
print("缺失:", missing or "无")
if missing:
    if shutil.which("cargo") is None:
        print("装 Rust 工具链...")
        sh("curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal")
    os.environ["PATH"] = os.path.expanduser("~/.cargo/bin") + ":" + os.environ["PATH"]
    print("装 sglang[srt] + aiohttp + torchvision + " + TF_PIN + "（约 5-10 分钟）...")
    r = sh([sys.executable, "-m", "pip", "install", "-q",
            "sglang[srt]", "aiohttp", "torchvision", TF_PIN])
    print("退出码:", r.returncode)
    if r.returncode:
        print(r.stdout[-2500:]); print(r.stderr[-2500:])
else:
    print("三个包都在，跳过安装。")

print()
for pkg in ["kernels", "torchaudio"]:
    u = sh([sys.executable, "-m", "pip", "uninstall", "-y", "-q", pkg])
    print("  卸 %-12s 退出码 %s" % (pkg, u.returncode))
if ver("transformers") != "5.12.1":
    sh([sys.executable, "-m", "pip", "install", "-q", TF_PIN])
print()
for p in CHECK + ["transformers", "torch"]:
    print("  %-14s %s" % (p, ver(p) or "（未安装）"))
print("  %-14s %s" % ("kernels", ver("kernels") or "已卸载 OK"))
print("  %-14s %s" % ("torchaudio", ver("torchaudio") or "已卸载 OK"))


## 2. 写出脚本（与 2026-09-05 那轮一个字节不改）

In [ ]:
import io, hashlib
files = {}
files["bench_serving.py"] = '# -*- coding: utf-8 -*-\n"""vLLM 服务端压测：并发扫描下的吞吐 / TTFT / TPOT，以及前缀复用的效果。\n\n指标定义（与 JD 里那套一致）：\n  TTFT  Time To First Token   —— 首 token 延迟，决定交互体感\n  TPOT  Time Per Output Token —— 首 token 之后的平均出词间隔\n  吞吐   总输出 token 数 / 墙钟时间\n\n用法（先 bash serve.sh 起服务）：\n  python bench_serving.py                 # 并发扫描\n  python bench_serving.py --prefix-test   # 前缀复用对照\n"""\nimport argparse, asyncio, json, statistics as st, time\nimport aiohttp\n\nURL = "http://127.0.0.1:8000/v1/chat/completions"\nMODEL = "Qwen/Qwen2.5-0.5B-Instruct"\n\n# 一段较长的共享 system prompt：开 --enable-prefix-caching 后其 prefill 只算一次\nSHARED_PREFIX = (\n    "You are a meticulous technical assistant. Answer concisely and precisely. "\n    "Always reason step by step before answering. " * 20\n)\n\n\nasync def one_request(sess, prompt, max_tokens, use_prefix):\n    msgs = ([{"role": "system", "content": SHARED_PREFIX}] if use_prefix else []) + \\\n           [{"role": "user", "content": prompt}]\n    body = {"model": MODEL, "messages": msgs, "max_tokens": max_tokens,\n            "temperature": 0.0, "stream": True}\n    t0 = time.perf_counter()\n    ttft, n_tok, last = None, 0, t0\n    async with sess.post(URL, json=body) as resp:\n        async for raw in resp.content:\n            line = raw.decode("utf-8").strip()\n            if not line.startswith("data: ") or line == "data: [DONE]":\n                continue\n            delta = json.loads(line[6:])["choices"][0].get("delta", {})\n            if delta.get("content"):\n                now = time.perf_counter()\n                if ttft is None:\n                    ttft = now - t0\n                n_tok += 1\n                last = now\n    return dict(ttft=ttft or 0.0, total=last - t0, n_tok=n_tok)\n\n\nasync def run_batch(n_conc, n_req, max_tokens, use_prefix):\n    prompts = [f"Explain concept #{i} in distributed systems." for i in range(n_req)]\n    sem = asyncio.Semaphore(n_conc)\n\n    async def guarded(sess, p):\n        async with sem:\n            return await one_request(sess, p, max_tokens, use_prefix)\n\n    timeout = aiohttp.ClientTimeout(total=600)\n    async with aiohttp.ClientSession(timeout=timeout) as sess:\n        await one_request(sess, "warmup", 4, use_prefix)          # 预热\n        t0 = time.perf_counter()\n        rs = await asyncio.gather(*(guarded(sess, p) for p in prompts))\n        wall = time.perf_counter() - t0\n\n    tot_tok = sum(r["n_tok"] for r in rs)\n    tpots = [(r["total"] - r["ttft"]) / max(r["n_tok"] - 1, 1) for r in rs if r["n_tok"] > 1]\n    return dict(conc=n_conc, wall=wall, tput=tot_tok / wall, rps=len(rs) / wall,\n                ttft_p50=st.median(r["ttft"] for r in rs),\n                ttft_p99=sorted(r["ttft"] for r in rs)[int(len(rs) * 0.99) - 1],\n                tpot_p50=st.median(tpots) if tpots else 0.0, tot_tok=tot_tok)\n\n\nasync def sweep(args):\n    print(f"{\'并发\':>5}{\'请求\':>6}{\'墙钟s\':>9}{\'吞吐 tok/s\':>13}{\'RPS\':>8}"\n          f"{\'TTFT p50\':>11}{\'TTFT p99\':>11}{\'TPOT p50\':>11}")\n    print("-" * 74)\n    out = []\n    for c in [1, 2, 4, 8, 16, 32]:\n        r = await run_batch(c, max(c * 4, 16), args.max_tokens, use_prefix=False)\n        print(f"{r[\'conc\']:>5}{max(c*4,16):>6}{r[\'wall\']:>9.2f}{r[\'tput\']:>13.1f}"\n              f"{r[\'rps\']:>8.2f}{r[\'ttft_p50\']*1e3:>10.1f}ms{r[\'ttft_p99\']*1e3:>10.1f}ms"\n              f"{r[\'tpot_p50\']*1e3:>10.2f}ms")\n        out.append(r)\n    json.dump(out, open("sweep_results.json", "w"), indent=1)\n    base = out[0]["tput"]\n    print(f"\\ncontinuous batching 收益：并发 1 → 32，吞吐 "\n          f"{base:.1f} → {out[-1][\'tput\']:.1f} tok/s（{out[-1][\'tput\']/base:.1f}×），"\n          f"TTFT p50 {out[0][\'ttft_p50\']*1e3:.0f} → {out[-1][\'ttft_p50\']*1e3:.0f} ms")\n    print("吞吐与延迟的取舍就在这张表里：并发拉高吞吐涨，但 TTFT 同步恶化。")\n\n\nasync def prefix_test(args):\n    print("前缀复用对照（服务端需带 --enable-prefix-caching 启动）")\n    print(f"{\'场景\':<26}{\'吞吐 tok/s\':>13}{\'TTFT p50\':>12}")\n    print("-" * 51)\n    for label, up in [("无共享前缀", False), (f"共享前缀 ({len(SHARED_PREFIX)} 字符)", True)]:\n        r = await run_batch(8, 32, args.max_tokens, use_prefix=up)\n        print(f"{label:<26}{r[\'tput\']:>13.1f}{r[\'ttft_p50\']*1e3:>11.1f}ms")\n    print("\\n共享前缀命中 KV cache 后，重复的 prefill 不再重算，TTFT 应显著下降。")\n    print("对比未开 --enable-prefix-caching 重启服务再跑一次，差值即为该特性的真实收益。")\n\n\nif __name__ == "__main__":\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--max-tokens", type=int, default=128)\n    ap.add_argument("--prefix-test", action="store_true")\n    a = ap.parse_args()\n    asyncio.run(prefix_test(a) if a.prefix_test else sweep(a))\n'
files["isolate.py"] = '# -*- coding: utf-8 -*-\n"""隔离测试：一次性发 N 条并发请求，期间无新请求到达。\n用于区分「稳态 batch=N 解码慢」与「新请求 prefill 插队拖累解码」。"""\nimport asyncio, sys, time, json, statistics as st\nimport aiohttp\nURL="http://127.0.0.1:8000/v1/chat/completions"; MODEL="Qwen/Qwen2.5-0.5B-Instruct"\n\nasync def one(s, i, n_tok):\n    b={"model":MODEL,"messages":[{"role":"user","content":f"Explain idea {i} briefly."}],\n       "max_tokens":n_tok,"temperature":0.0,"stream":True}\n    t0=time.perf_counter(); ttft=None; n=0; last=t0\n    async with s.post(URL,json=b) as r:\n        async for raw in r.content:\n            l=raw.decode().strip()\n            if not l.startswith("data: ") or l=="data: [DONE]": continue\n            d=json.loads(l[6:])["choices"][0].get("delta",{})\n            if d.get("content"):\n                now=time.perf_counter()\n                if ttft is None: ttft=now-t0\n                n+=1; last=now\n    return (ttft or 0), last-t0, n\n\nasync def burst(n, n_tok=64):\n    """严格同时发 n 条，全部跑完才结束 —— 稳态就是 batch=n。"""\n    async with aiohttp.ClientSession(timeout=aiohttp.ClientTimeout(total=600)) as s:\n        await one(s,-1,4)                       # 预热\n        t0=time.perf_counter()\n        rs=await asyncio.gather(*(one(s,i,n_tok) for i in range(n)))\n        wall=time.perf_counter()-t0\n    tp=[(tot-tt)/max(k-1,1) for tt,tot,k in rs if k>1]\n    tot=sum(k for _,_,k in rs)\n    print(f"  一次性 {n:>2} 条并发: 墙钟 {wall:>6.2f}s  总吞吐 {tot/wall:>6.1f} tok/s  "\n          f"TPOT p50 {st.median(tp)*1e3:>7.2f} ms")\n\nasync def main():\n    print("=== 无新到达的纯稳态测试 ===")\n    for n in [1, 2, 3, 4, 8, 16]:\n        await burst(n)\n\nasyncio.run(main())\n'
for n, s in files.items():
    io.open(n, "w", encoding="utf-8").write(s)
    print("写出 %-18s %5d 字符  sha=%s" % (n, len(s), hashlib.sha256(s.encode()).hexdigest()[:16]))
print()
print("bench_serving.py 的 sha 应与 2026-09-05 那轮一致: ef5bf10717ecdacf")


## 3. 启动器

与那轮同样的参数：`--attention-backend triton --sampling-backend pytorch`、
`--context-length 2048`、`--mem-fraction-static 0.80`。
**服务只起一次**，三轮压测都打同一个服务——这样轮次之间的差异才只来自测量本身。


In [ ]:
import subprocess, sys, time, requests

MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

def serve_sglang(extra, tag, wait=420):
    subprocess.run(["pkill", "-f", "sglang.launch_server"], check=False)
    time.sleep(10)
    cmd = [sys.executable, "-m", "sglang.launch_server",
           "--model-path", MODEL, "--host", "127.0.0.1", "--port", "8000",
           "--context-length", "2048", "--mem-fraction-static", "0.80",
           "--attention-backend", "triton", "--sampling-backend", "pytorch"] + extra
    print("启动参数:", " ".join(cmd[2:]))
    lg = open("/content/rp_%s.log" % tag, "w")
    p = subprocess.Popen(cmd, stdout=lg, stderr=subprocess.STDOUT)
    for i in range(wait // 2):
        if p.poll() is not None:
            print("[%s] 退出码 %s" % (tag, p.returncode))
            print(open("/content/rp_%s.log" % tag).read()[-2500:])
            return None
        try:
            if requests.get("http://127.0.0.1:8000/v1/models", timeout=2).status_code == 200:
                print("[%s] 就绪，用时 %ds" % (tag, i * 2))
                return p
        except requests.RequestException:
            pass
        time.sleep(2)
    print("[%s] 超时" % tag)
    print(open("/content/rp_%s.log" % tag).read()[-2500:])
    return None

proc = serve_sglang([], "radix_on")


## 4. 并发扫描 × 3 轮

同一个服务、同一份脚本，连跑三次。


In [ ]:
import subprocess, sys, json, os

RUNS = []
for k in range(3):
    print("========== 第 %d 轮 ==========" % (k + 1))
    r = subprocess.run([sys.executable, "-u", "bench_serving.py"],
                       capture_output=True, text=True)
    print(r.stdout)
    if os.path.exists("sweep_results.json"):
        RUNS.append(json.load(open("sweep_results.json")))
        os.rename("sweep_results.json", "rp_sweep_%d.json" % k)
    else:
        print("  本轮没拿到结果:", r.stderr[-800:])
print("拿到", len(RUNS), "轮扫描数据")


## 5. 稳态测试 × 3 轮

In [ ]:
import subprocess, sys, re

ISO_RUNS = []
for k in range(3):
    print("========== 稳态 第 %d 轮 ==========" % (k + 1))
    r = subprocess.run([sys.executable, "-u", "isolate.py"], capture_output=True, text=True)
    print(r.stdout)
    d = {}
    for m in re.finditer(r"一次性\s*(\d+)\s*条并发:\s*墙钟\s*([\d.]+)s\s*总吞吐\s*([\d.]+)\s*tok/s\s*TPOT p50\s*([\d.]+)", r.stdout):
        d[int(m.group(1))] = {"wall": float(m.group(2)), "tput": float(m.group(3)), "tpot": float(m.group(4))}
    ISO_RUNS.append(d)
print("拿到", len([d for d in ISO_RUNS if d]), "轮稳态数据")


## 6. 判定 —— 并发扫描

In [ ]:
import statistics as st

if not RUNS:
    print("没有扫描数据。")
else:
    print("%-6s %30s %10s %10s" % ("并发", "三轮吞吐 tok/s", "中位", "极差比"))
    print("-" * 62)
    med = {}
    for c in [1, 2, 4, 8, 16, 32]:
        vals = []
        for run in RUNS:
            for row in run:
                if row["conc"] == c:
                    vals.append(row["tput"])
        if not vals:
            continue
        m = st.median(vals)
        med[c] = m
        spread = max(vals) / min(vals) if min(vals) > 0 else float("inf")
        print("%-6d %30s %10.1f %9.2f×" % (c, " / ".join("%.1f" % v for v in vals), m, spread))

    print()
    print("=== 并发 16 是不是真实凹陷 ===")
    if 16 in med and 8 in med and 32 in med:
        print("  中位: 并发8 = %.1f, 并发16 = %.1f, 并发32 = %.1f" % (med[8], med[16], med[32]))
        dip = med[16] < med[8] and med[16] < med[32]
        vals16 = [row["tput"] for run in RUNS for row in run if row["conc"] == 16]
        low = sum(1 for v in vals16 if v < min(med[8], med[32]))
        print("  三轮中低于两侧中位的轮次数: %d / %d" % (low, len(vals16)))
        print()
        if dip and low == len(vals16):
            print("  判定：**真实凹陷** —— 三轮全部低于两侧，可复现。")
            print("        这比一条漂亮的单调曲线更值钱，值得单独解释。")
        elif low == 0:
            print("  判定：**抖动** —— 三轮都没再复现。")
            print("        2026-09-05 那个 496.0 是噪声，结论里应剔除并说明。")
        else:
            print("  判定：**间歇性** —— %d/%d 轮出现。" % (low, len(vals16)))
            print("        不能当稳定现象讲，也不能说它不存在；如实写「间歇出现，未定性」。")


## 7. 判定 —— 稳态 3 条

In [ ]:
import statistics as st

good = [d for d in ISO_RUNS if d]
if not good:
    print("没有稳态数据。")
else:
    print("%-8s %26s %26s" % ("并发条数", "三轮吞吐 tok/s", "三轮墙钟 s"))
    print("-" * 64)
    for n in [1, 2, 3, 4, 8, 16]:
        tp = [d[n]["tput"] for d in good if n in d]
        wl = [d[n]["wall"] for d in good if n in d]
        if not tp:
            continue
        print("%-8d %26s %26s" % (n,
              " / ".join("%.1f" % v for v in tp),
              " / ".join("%.2f" % v for v in wl)))

    print()
    print("=== 稳态 3 条是不是真实的 ===")
    tp3 = [d[3]["tput"] for d in good if 3 in d]
    nb  = [st.median([d[n]["tput"] for d in good if n in d]) for n in [2, 4] if any(n in d for d in good)]
    if tp3 and nb:
        base = min(nb)
        slow = sum(1 for v in tp3 if v < base / 3)
        print("  三轮 3 条吞吐: %s ；邻居(2/4条)中位最小值 = %.1f" % (
              " / ".join("%.1f" % v for v in tp3), base))
        print("  明显偏慢(低于邻居 1/3)的轮次: %d / %d" % (slow, len(tp3)))
        print()
        if slow == len(tp3):
            print("  判定：**可复现** —— 每轮都塌。TPOT 正常而墙钟长，")
            print("        方向指向排队/调度而非解码，值得往下查。")
        elif slow == 0:
            print("  判定：**抖动** —— 三轮都没再出现。")
            print("        2026-09-05 那个 22.1 是噪声，结论里应剔除并说明。")
        else:
            print("  判定：**间歇性** —— %d/%d 轮出现，如实写「间歇出现，未定性」。" % (slow, len(tp3)))


## 8. 边界

- 三轮打的是**同一个服务实例**：这隔离了「启动差异」，但**没有**隔离
  Colab 宿主机上的邻居干扰——那正是我们怀疑的噪声来源之一。
  若判定为抖动，严格说是「同一实例内不可复现」，不能断言原因就是邻居干扰。
- 三轮仍是很小的样本。判为「间歇性」时不要硬凑解释。
- 与 2026-09-05 那轮的可比性：同硬件、同模型、同参数、`bench_serving.py` 的 sha 已在第 2 节比对。
